# AP → TI → AP relay (128 elements)
Walk through the four runners, one cell each. Works locally and in Colab (see the first cell).

In [ ]:
# --- Step 0: find the project root (works from any folder: VS Code, Jupyter, Colab) and add every module to sys.path
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "integration" / "03_full_pipeline" / "_paths.py").exists()), None)
if ROOT is None:
    raise SystemExit("Project root not found. Open the NITK-UsoundSim-v2 folder in VS Code (File > Open Folder) and run again.")
sys.path.insert(0, str(ROOT / "integration" / "03_full_pipeline"))
import _paths                                   # noqa: F401  (adds modules/01..05 and integration/01..03 to sys.path)
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline") if "ipykernel" in sys.modules else None
import matplotlib.pyplot as plt
import numpy as np
print("project root:", ROOT)
from usoundsim_relay import run_relay, RelayConfig
from usoundsim_relay.verify import das_image
from nitk_usoundsim import config
print('elements:', config.N_ELEMENTS, '| f0 MHz:', config.F0/1e6, '| pitch mm:', round(config.PITCH*1e3,3))

## Run all four runners
Nine wire targets: x, z in mm and amplitude. Swap in `"generic"`, `"abdomen"`, a points CSV or a phantom image.

In [ ]:
pts = np.array([[x, z, 1.0] for z in (10, 20, 30) for x in (-6, 0, 6)])
res = run_relay(pts, RelayConfig(), units="mm")

## Runner 2 and 3: what one scatterer sees
Incident pressure from Acoustic Propagation, then the reflected pressure Tissue Interaction hands back.

In [ ]:
k = 4; t = res.t_axis*1e6
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(t, res.incident[k]);  ax[0].set_title("AP part 1: incident"); ax[0].set_xlabel("time [us]")
ax[1].plot(t, res.reflected[k], c="r"); ax[1].set_title("TI stage 2: reflected"); ax[1].set_xlabel("time [us]")
for a in ax: a.set_xlim(t[np.abs(res.incident[k])>1e-3*np.abs(res.incident[k]).max()][[0,-1]]+[-2,2])
plt.show()

## Runner 4: the baton for Receiver Beamforming
`received_pressure` has one row per receive element (128) and one column per time sample.

In [ ]:
rx = res.received_pressure; print("received_pressure:", rx.shape)
v = np.percentile(np.abs(rx), 99.5)
plt.figure(figsize=(7,4)); plt.imshow(rx.T, aspect="auto", cmap="seismic", vmin=-v, vmax=v,
    extent=[0, 128, res.t_axis[-1]*1e6, res.t_axis[0]*1e6]); plt.xlabel("RX element"); plt.ylabel("time [us]"); plt.show()

## Quality check: do the targets focus where we put them?
Quick delay-and-sum for checking only. The Receiver BF team writes the real beamformer.

In [ ]:
xs = np.linspace(-10e-3, 10e-3, 300); zs = np.linspace(2e-3, 33e-3, 300)
img = das_image(rx, res.t_axis, res.element_x, xs, zs)
plt.figure(figsize=(4,6)); plt.imshow(20*np.log10(np.maximum(img/img.max(),1e-6)), vmin=-50, vmax=0, cmap="gray",
    extent=[-10,10,33,2]); plt.xlabel("x [mm]"); plt.ylabel("z [mm]"); plt.show()

## Save the hand-over files
`outdir` writes `01_` to `04_` plus `receiver_bf_input.npz`.

In [ ]:
res = run_relay(pts, RelayConfig(), units="mm", outdir="outputs/notebook_run", verbose=False)
print(res.files)